# 1. What you'll learn

        - read a GP posterior mean and uncertainty band
- see how kernel length scale encodes smoothness
- run expected-improvement Bayesian optimization against random search

        **The one question this notebook answers:** How can a model say both what it predicts and where it knows too little, then use that uncertainty to choose the next experiment?

# 2. Setup

A one-dimensional objective keeps every probability visible. Scikit-learn supplies exact Gaussian-process regression; SciPy supplies the normal CDF and density used by expected improvement.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.metrics import mean_squared_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The hidden objective is a smooth wavy bowl. We begin with five noisy evaluations. x is the controllable setting; y is the costly measured loss we want to minimize.

**Small example:** Suppose five losses are [4.0, 1.8, 0.9, 1.2, 3.5]. The current best is 0.9. A candidate predicting 0.8±0.05 is safe; one predicting 1.0±0.8 may still have larger expected improvement because uncertainty leaves room below 0.9.

In [ ]:
def objective(x): return .18*(x-1.6)**2+np.sin(2.7*x)+.15*np.cos(7*x)
rng=np.random.default_rng(SEED);grid=np.linspace(-2.5,4.5,500)[:,None]
x_obs=np.array([-2.2,-.7,.8,2.5,4.2])[:,None];y_obs=objective(x_obs[:,0])+rng.normal(0,.08,len(x_obs))
true_y=objective(grid[:,0]);true_x=grid[np.argmin(true_y),0]
print("Observed shape:",x_obs.shape,"current best:",round(y_obs.min(),3),"teaching-only true minimum x:",round(true_x,3))

# 4. Explore

We inspect the true landscape for teaching, the sparse initial observations available to the model, and kernel correlations at three length scales.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.6));axes[0].plot(grid,true_y);axes[0].set_title("Hidden objective")
axes[1].plot(grid,true_y,color=".8");axes[1].scatter(x_obs,y_obs,color="#176b66",s=45);axes[1].set_title("Only five costly observations")
d=np.linspace(0,3,150);[axes[2].plot(d,np.exp(-d**2/(2*l*l)),label=f"length={l}") for l in [.2,.7,1.5]];axes[2].set(title="RBF-style correlation",xlabel="distance",ylabel="similarity");axes[2].legend();plt.tight_layout();plt.show()

**Takeaway:** The objective has several local dips, so blindly descending from one point can miss the best basin.

**Takeaway:** Five observations leave large gaps where honest uncertainty should widen.

**Takeaway:** A short length scale permits rapid wiggles; a long one couples distant x values and produces smoother beliefs.

# 5. Prepare

There is no conventional random split in the optimization loop: each queried point becomes training data. Both optimizers receive the same five initial noisy observations and a budget of eight further measurements. The dense grid of true values is available only because this is a synthetic teaching problem; it scores the methods after queries are chosen, never supplies measured values to the optimizer.

The kernel describes smoothness and observation noise. With `WhiteKernel`, scikit-learn's returned standard deviation includes a fresh observation's noise as well as uncertainty about the latent function. A narrow predictive band is conditional on the chosen kernel and fitted parameters; it is not a guarantee of calibration.

In [ ]:
kernel=ConstantKernel(1.0,(.1,10))*Matern(length_scale=.8,length_scale_bounds=(.1,3),nu=2.5)+WhiteKernel(.01,(1e-5,.2))
def fit_gp(x,y): return GaussianProcessRegressor(kernel=kernel,normalize_y=True,n_restarts_optimizer=1,random_state=SEED).fit(x,y)
gp=fit_gp(x_obs,y_obs);print("Initial kernel learned:",gp.kernel_)

# 6. Train

Conditioning a GP turns prior correlations into a posterior mean and variance. Bayesian optimization refits that posterior after every noisy measurement and queries where expected improvement balances predicted loss with uncertainty.

1. Build a covariance matrix from distances and the kernel.
2. Condition the joint Gaussian on observed y values.
3. Predict mean μ and standard deviation σ over candidates.
4. Maximize expected improvement, measure that x, append it, and repeat.

This is a small **plug-in expected-improvement demonstration**: the incumbent is the lowest noisy measurement and the supplied uncertainty includes observation noise. A lucky low measurement can therefore distort the acquisition. It is not a noise-aware EI implementation; substantial noise calls for an acquisition that integrates uncertainty in the incumbent rather than treating that measurement as exact. The later score avoids rewarding lucky noise by evaluating the latent objective at all queried points.

In [ ]:
def expected_improvement(mu, sigma, best):
    gain = best - mu - .01
    z = np.divide(gain, sigma, out=np.zeros_like(gain), where=sigma > 1e-12)
    result = gain * norm.cdf(z) + sigma * norm.pdf(z)
    return np.where(sigma > 1e-12, result, np.maximum(gain, 0))

N_NEW = 8
NOISE_SD = .08
x_bo, y_bo = x_obs.copy(), y_obs.copy()
history = []
for step in range(N_NEW):
    model = fit_gp(x_bo, y_bo)
    mean, std = model.predict(grid, return_std=True)
    ei = expected_improvement(mean, std, y_bo.min())
    x_new = grid[int(np.argmax(ei)):int(np.argmax(ei)) + 1]
    y_new = objective(x_new[:, 0]) + rng.normal(0, NOISE_SD, 1)
    x_bo = np.vstack([x_bo, x_new])
    y_bo = np.r_[y_bo, y_new]
    history.append((float(x_new[0, 0]), float(y_new[0])))
print("Noisy queries:", [(round(x, 2), round(y, 2)) for x, y in history])
print("Best noisy measurement (not the comparison score):", round(y_bo.min(), 3))

# 7. Evaluate on the same basis and budget

We score **latent best-so-far loss at queried locations** for both methods. Each receives the same five starting points and eight new queries; both measurement streams have Gaussian noise with standard deviation 0.08. BO uses its noisy stream to adapt; random search does not use measurements to select the next point. Neither method receives latent values during selection.

Because this toy objective is known, latent values let us compare which locations were found without treating a lucky noise draw as a better solution. Real experiments need a separate equal-budget remeasurement protocol or another defensible estimate; they cannot read a hidden objective.

The curve shows one seeded BO run against 50 random runs. The random ribbon is the 10th–90th percentile of those runs, **not a confidence interval**. One BO seed does not establish general superiority. Posterior RMSE and the constant-mean baseline both use the same final BO observations. The plotted ±1.96σ band is a model-based predictive interval including observation noise, not an empirically calibrated 95% claim.

In [ ]:
final_gp = fit_gp(x_bo, y_bo)
mu, std = final_gp.predict(grid, return_std=True)
baseline = np.repeat(y_bo.mean(), len(grid))
print(f"GP grid RMSE={mean_squared_error(true_y, mu)**.5:.3f}; "
      f"same-data mean baseline={mean_squared_error(true_y, baseline)**.5:.3f}")

n_initial = len(x_obs)
initial_latent = objective(x_obs[:, 0])
bo_curve = np.minimum.accumulate(objective(x_bo[:, 0]))[n_initial:]
random_curves, random_measurements = [], []
for seed in range(50):
    rr = np.random.default_rng(1000 + seed)
    # Choose all locations without consulting either measurements or truth.
    xs = rr.uniform(-2.5, 4.5, N_NEW)
    noisy_ys = objective(xs) + rr.normal(0, NOISE_SD, N_NEW)
    random_measurements.append(np.r_[y_obs, noisy_ys])
    latent_at_queries = np.r_[initial_latent, objective(xs)]
    random_curves.append(np.minimum.accumulate(latent_at_queries)[n_initial:])
random_curves = np.asarray(random_curves)
random_mean = random_curves.mean(axis=0)
random_low, random_high = np.quantile(random_curves, [.1, .9], axis=0)
assert len(x_bo) == n_initial + N_NEW
assert random_curves.shape == (50, N_NEW)
assert all(len(values) == len(y_bo) for values in random_measurements)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
axes[0].plot(grid, mu, label="posterior mean")
axes[0].fill_between(grid[:, 0], mu - 1.96 * std, mu + 1.96 * std,
                     alpha=.25, label="model predictive ±1.96σ")
axes[0].scatter(x_bo, y_bo, s=28, color="#176b66")
axes[0].plot(grid, true_y, "--", color="black", alpha=.55, label="latent truth")
axes[0].set_title("Posterior after equal-budget queries")
axes[0].legend(fontsize=7)
steps = np.arange(1, N_NEW + 1)
axes[1].plot(steps, bo_curve, marker="o", label="BO: one seeded run")
axes[1].plot(steps, random_mean, marker="o", label="random: 50-run mean")
axes[1].fill_between(steps, random_low, random_high, alpha=.2,
                     label="random: 10–90% spread")
axes[1].axhline(true_y.min(), ls="--", color="black", label="grid minimum")
axes[1].set(title="Latent best loss at queried points", xlabel="new evaluations")
axes[1].legend(fontsize=7)
axes[2].plot(grid, std, color="#d38b45")
axes[2].scatter(x_bo, np.zeros(len(x_bo)), marker="|", color="black")
axes[2].set(title="Predictive uncertainty (includes noise)", xlabel="x", ylabel="σ")
plt.tight_layout()
plt.savefig("key_figure.png", dpi=160, bbox_inches="tight")
plt.show()

**Read the result:** Compare BO's latent loss with the random-run spread at the same number of new evaluations. If it finds a low value early, that is evidence about this objective and seed, not a general speedup claim. Predictive uncertainty often contracts near observations but retains an observation-noise floor. Changing smoothness, initial points, or noise can reverse the comparison.

# 8. Break it: hold the observations fixed

Fit an excessively long, fixed length scale and almost no observation noise to **the same final `x_bo, y_bo` used by the tuned GP**. This isolates kernel assumptions rather than confounding them with the number or location of observations. Compare grid RMSE and band width.

We also print the fraction of this deterministic grid whose latent truth lies inside each model's predictive band. That is a descriptive truth-containment check on one function, **not repeated-sampling coverage or a calibration test**. The band includes observation noise, which differs from uncertainty about the latent function alone.

In the checked seed, the over-smooth model has **lower RMSE but much worse truth containment**. Do not force the experiment to tell a simpler winner/loser story: point error and uncertainty quality differ. The fitted kernel can also hit parameter bounds and emit convergence warnings; inspect those warnings and repeat with other seeds before trusting its uncertainty.

In [ ]:
bad_kernel = (ConstantKernel(1.0, constant_value_bounds="fixed")
              * Matern(length_scale=8, length_scale_bounds="fixed", nu=2.5)
              + WhiteKernel(1e-6, noise_level_bounds="fixed"))
bad = GaussianProcessRegressor(kernel=bad_kernel, optimizer=None,
                               normalize_y=True, random_state=SEED).fit(x_bo, y_bo)
bad_mu, bad_std = bad.predict(grid, return_std=True)
bad_inside = np.mean(np.abs(true_y - bad_mu) <= 1.96 * bad_std)
good_inside = np.mean(np.abs(true_y - mu) <= 1.96 * std)
print(f"Same observations: {len(x_bo)} for each model")
print(f"Grid RMSE — over-smooth: {mean_squared_error(true_y, bad_mu)**.5:.3f}; "
      f"tuned: {mean_squared_error(true_y, mu)**.5:.3f}")
print(f"Mean predictive σ — over-smooth: {bad_std.mean():.3f}; tuned: {std.mean():.3f}")
print(f"Descriptive latent-truth containment — over-smooth: {bad_inside:.3f}; "
      f"tuned: {good_inside:.3f} (not a calibration estimate)")

**Use this when…** evaluations are expensive, data are small, and a credible smoothness model can guide the next measurement.

**Don't use this when…** data are huge, the objective changes abruptly, many dimensions are irrelevant, or random/grid search can evaluate a cheap objective freely. Do not assume a GP's narrow band is calibrated simply because the model is probabilistic.

## Try this

1. Repeat the complete experiment across at least 20 initial/noise seeds and compare distributions, retaining equal query budgets and the same score for both methods.
2. Start with three clustered points and inspect where predictive uncertainty grows. Preserve the same start for both search methods.
3. Replace Matern ν=2.5 with RBF; compare both kernels on identical observations.
4. Raise observation noise to 0.4. Inspect how the noisy incumbent affects plug-in EI; this implementation need not explore sensibly under that change.